[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/exercises/20_clustering/90_exercises.ipynb)

# Bài tập — Chương 20: Clustering (L0 → L7)

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/exercises/20_clustering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, HDBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

rng = np.random.default_rng(0)

## L0 — Nhớ khái niệm

**Câu 1.** Viết hàm mục tiêu K-Means và hai bước của thuật toán Lloyd; công thức silhouette của một điểm; định nghĩa core point trong DBSCAN; 4 kiểu linkage.

✍️ Trả lời:

**Đáp án:** $J = \sum_i \lVert\mathbf x_i - \boldsymbol\mu_{c_i}\rVert^2$; (1) gán $c_i = \arg\min_k\lVert\mathbf x_i - \boldsymbol\mu_k\rVert^2$, (2) $\boldsymbol\mu_k$ = trung bình các điểm của cụm $k$. $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$. Core point: có ≥ `min_samples` điểm (kể cả chính nó) trong bán kính ε. Linkage: single (min), complete (max), average (trung bình), Ward (mức tăng tổng bình phương trong cụm).

## L1 — Giải thích bằng lời

**Câu 2.** Trưởng phòng marketing hỏi: "K-Means chia khách hàng thành 5 nhóm — vậy có đúng 5 loại khách hàng không?" Trả lời thế nào cho chính xác và hữu ích?

✍️ Trả lời:

**Gợi ý:** K-Means **luôn** chia thành đúng số nhóm ta yêu cầu, kể cả khi dữ liệu không có nhóm rõ ràng — "5" là lựa chọn của người phân tích, không phải phát hiện. Các chỉ số (silhouette, elbow) chỉ gợi ý; kết quả phụ thuộc feature được chọn, cách scale, và giả định cụm "tròn". Câu hỏi hữu ích hơn: các nhóm có **khác nhau rõ** về hành vi không, có **ổn định** khi chạy lại trên dữ liệu tháng sau không, có **hành động** marketing riêng cho từng nhóm không? Nếu có — 5 nhóm là một cách phân khúc hữu ích, không phải "sự thật".

## L2 — Tính tay

**Câu 3.** 1D: điểm $x = [1, 2, 3, 10, 11, 30]$, tâm ban đầu $\mu = [1, 11]$. Thực hiện **một** vòng Lloyd: gán, cập nhật tâm, tính inertia trước và sau cập nhật.

In [ ]:
x = np.array([1, 2, 3, 10, 11, 30.0])
mu0 = np.array([1, 11.0])
### START CODE HERE ### (≈ 5 dòng)
labels = np.argmin(np.abs(x[:, None] - mu0[None]), axis=1)
inertia_before = np.sum((x - mu0[labels]) ** 2)
mu1 = np.array([x[labels == k].mean() for k in range(2)])
inertia_after = np.sum((x - mu1[labels]) ** 2)
### END CODE HERE ###
print(labels, mu1, inertia_before, inertia_after)

In [ ]:
check_equal(labels.tolist(), [0, 0, 0, 1, 1, 1], "gán: 1, 2, 3 về tâm 1; 10, 11, 30 về tâm 11")
check_close(mu1, [2, 17], message="tâm mới")
check_close(inertia_before, 0 + 1 + 4 + 1 + 0 + 361, message="inertia trước")
check_close(inertia_after, 1 + 0 + 1 + 49 + 36 + 169, message="inertia sau cập nhật (giảm)")

**Câu 4.** Silhouette của điểm $x = 3$ trong cách chia $\{1, 2, 3\}, \{10, 11, 30\}$ (khoảng cách tuyệt đối 1D).

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
a3 = np.mean([abs(3 - 1), abs(3 - 2)])
b3 = np.mean([abs(3 - 10), abs(3 - 11), abs(3 - 30)])
s3 = (b3 - a3) / max(a3, b3)
### END CODE HERE ###

In [ ]:
check_close((a3, b3), (1.5, 14.0), message="a(i), b(i)")
check_close(s3, 12.5 / 14, message="silhouette")

## L3 — Code

**Bài 5.** Viết `kmeans_pp_init(X, k, seed)` (k-means++) và `lloyd(X, centers, n_iter)` trả về `(centers, labels, inertia_history)`. Chứng minh inertia không tăng qua các vòng; so sánh với `KMeans` của sklearn.

In [ ]:
def kmeans_pp_init(X, k, seed=0):
    r = np.random.default_rng(seed)
    centers = [X[r.integers(len(X))]]
    ### START CODE HERE ### (≈ 3 dòng)
    for _ in range(1, k):
        d2 = np.min(((X[:, None, :] - np.array(centers)[None]) ** 2).sum(-1), axis=1)
        centers.append(X[r.choice(len(X), p=d2 / d2.sum())])
    ### END CODE HERE ###
    return np.array(centers)


def lloyd(X, centers, n_iter=100):
    history = []
    for _ in range(n_iter):
        ### START CODE HERE ### (≈ 6 dòng)
        # STARTER: # 1) bình phương khoảng cách mọi điểm–tâm; 2) labels = argmin; 3) history.append(inertia)
        # STARTER: # 4) tâm mới = trung bình mỗi cụm (giữ tâm cũ nếu rỗng); dừng nếu tâm không đổi
        d2 = ((X[:, None, :] - centers[None]) ** 2).sum(-1)
        labels = d2.argmin(1)
        history.append(d2[np.arange(len(X)), labels].sum())
        new = np.array([X[labels == j].mean(0) if np.any(labels == j) else centers[j] for j in range(len(centers))])
        if np.allclose(new, centers):
            break
        centers = new
        ### END CODE HERE ###
    return centers, labels, history

In [ ]:
X5, y5 = make_blobs(n_samples=500, centers=5, cluster_std=1.0, random_state=7)
c5, l5, h5 = lloyd(X5, kmeans_pp_init(X5, 5, seed=1))
check_true(np.all(np.diff(h5) <= 1e-9), "inertia không tăng")
sk5 = KMeans(5, n_init=10, random_state=0).fit(X5)
check_less(abs(h5[-1] - sk5.inertia_) / sk5.inertia_, 0.02, "inertia ≈ sklearn (trong 2%)")

**Bài 6.** Viết `elbow_and_silhouette(X, ks)` trả về DataFrame với cột `k`, `inertia`, `silhouette`; dùng nó trên dữ liệu 4 cụm và chọn K theo silhouette lớn nhất (`best_k`).

In [ ]:
def elbow_and_silhouette(X, ks):
    rows = []
    ### START CODE HERE ### (≈ 3 dòng)
    # STARTER: # với mỗi k: KMeans(k, n_init=10, random_state=0) → rows.append({"k", "inertia", "silhouette"})
    for k in ks:
        km = KMeans(k, n_init=10, random_state=0).fit(X)
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(X, km.labels_)})
    ### END CODE HERE ###
    return pd.DataFrame(rows)

In [ ]:
X6, _ = make_blobs(n_samples=600, centers=4, cluster_std=0.7, random_state=21)
table6 = elbow_and_silhouette(X6, range(2, 9))
best_k = int(table6.loc[table6.silhouette.idxmax(), "k"])
print(table6.round(3))
check_equal(best_k, 4, "silhouette chọn đúng K = 4")
check_true(table6.inertia.is_monotonic_decreasing, "inertia giảm khi K tăng")

## L4 — Debug

**Bài 7.** Đoạn sau phân cụm dữ liệu cảm biến (nhiệt độ °C, áp suất Pa, độ rung) nhưng kết quả chỉ phụ thuộc áp suất. Ngoài ra code chọn K bằng **inertia nhỏ nhất**. Sửa: chuẩn hoá feature, chọn K ∈ {2..6} bằng silhouette. Lưu `best_k7` và `ari7` (so với `y7`).

In [ ]:
n7 = 900
y7 = rng.integers(0, 3, n7)
temp = np.array([40, 60, 80])[y7] + rng.normal(0, 4, n7)
pressure = 101_325 + rng.normal(0, 3000, n7)                    # không mang thông tin nhóm, thang đo rất lớn
vib = np.array([0.2, 0.5, 0.2])[y7] + rng.normal(0, 0.05, n7)
X7 = np.c_[temp, pressure, vib]
### START CODE HERE ### (≈ 6 dòng)
# STARTER: inertias = {k: KMeans(k, n_init=10, random_state=0).fit(X7).inertia_ for k in range(2, 7)}     # 🐛 không scale
# STARTER: best_k7 = min(inertias, key=inertias.get)                                                     # 🐛 luôn chọn K lớn nhất
# STARTER: ari7 = adjusted_rand_score(y7, KMeans(best_k7, n_init=10, random_state=0).fit_predict(X7))
X7s = StandardScaler().fit_transform(X7)
sils = {k: silhouette_score(X7s, KMeans(k, n_init=10, random_state=0).fit_predict(X7s)) for k in range(2, 7)}
best_k7 = max(sils, key=sils.get)
ari7 = adjusted_rand_score(y7, KMeans(best_k7, n_init=10, random_state=0).fit_predict(X7s))
### END CODE HERE ###
print(f"K chọn = {best_k7} | ARI = {ari7:.3f}")

In [ ]:
check_equal(best_k7, 3, "chọn K = 3")
check_greater(ari7, 0.8, "ARI > 0.8")

## L5 — Phân tích

**Bài 8.** So sánh K-Means, Agglomerative (Ward), Agglomerative (single), DBSCAN (`eps` chọn bằng k-distance), HDBSCAN trên hai bộ dữ liệu: (a) 3 blobs có **phương sai khác nhau**, (b) hai **mặt trăng** có thêm 5% điểm nhiễu đều. Lưu ARI (chỉ tính trên các điểm không phải nhiễu thêm vào) vào DataFrame `ari_table` (hàng = thuật toán, cột = dữ liệu). Nhận xét.

In [ ]:
Xa, ya = make_blobs(600, centers=3, cluster_std=[0.4, 1.8, 0.4], random_state=8)
Xb0, yb0 = make_moons(500, noise=0.05, random_state=3)
Xb = np.vstack([Xb0, rng.uniform([-1.5, -1], [2.5, 1.5], (25, 2))])
### START CODE HERE ### (≈ 12 dòng)
# STARTER: # với mỗi bộ dữ liệu và 5 thuật toán: fit_predict, ARI trên len(yd) điểm đầu → results[thuật toán][dữ liệu]
# STARTER: # eps của DBSCAN: phân vị 95% khoảng cách tới láng giềng thứ 5 (k-distance)
# STARTER: ari_table = None
def kdist_eps(X, m=5, q=0.95):
    d, _ = NearestNeighbors(n_neighbors=m).fit(X).kneighbors(X)
    return np.quantile(d[:, -1], q)


results = {}
for dname, (Xd, yd, k) in {"(a) blobs phương sai khác": (StandardScaler().fit_transform(Xa), ya, 3),
                           "(b) mặt trăng + nhiễu": (Xb, yb0, 2)}.items():
    algos = {"K-Means": KMeans(k, n_init=10, random_state=0), "Ward": AgglomerativeClustering(k),
             "Single": AgglomerativeClustering(k, linkage="single"), "DBSCAN": DBSCAN(eps=kdist_eps(Xd), min_samples=5),
             "HDBSCAN": HDBSCAN(min_cluster_size=20)}
    for aname, model in algos.items():
        lab = model.fit_predict(Xd)
        results.setdefault(aname, {})[dname] = adjusted_rand_score(yd, lab[: len(yd)])
ari_table = pd.DataFrame(results).T
### END CODE HERE ###
ari_table.round(3)

In [ ]:
check_greater(ari_table.loc["DBSCAN", "(b) mặt trăng + nhiễu"], ari_table.loc["K-Means", "(b) mặt trăng + nhiễu"] + 0.3,
              "DBSCAN thắng K-Means trên mặt trăng")
check_greater(ari_table.loc["HDBSCAN", "(b) mặt trăng + nhiễu"], 0.8, "HDBSCAN tách được mặt trăng")

✍️ Nhận xét:

**Gợi ý:** trên mặt trăng (không lồi + nhiễu), thuật toán dựa trên mật độ (DBSCAN/HDBSCAN) và single linkage thắng; K-Means/Ward cắt ngang hai mặt trăng; single linkage dễ bị điểm nhiễu "nối cầu". Trên blobs phương sai khác nhau, K-Means/Ward khá ổn nhưng gán sai vùng biên của cụm rộng; DBSCAN với một eps khó phù hợp cả cụm dày lẫn cụm thưa (có thể coi phần cụm thưa là nhiễu); HDBSCAN xử lý tốt hơn. Không có thuật toán tốt nhất cho mọi hình dạng.

## L6 — Thực tế

**Bài 9.** Phân khúc khách hàng từ `customer_segments.csv` (bỏ cột `true_segment` khi phân cụm): chuẩn hoá, chọn K trong {3..8} bằng silhouette, fit K-Means, tạo bảng **profile** (trung bình từng feature + số khách mỗi cụm) và đặt tên cho từng cụm (list `segment_names`). Cuối cùng so ARI với `true_segment` (`ari9`) — mục tiêu ≥ 0.85.

In [ ]:
seg = pd.read_csv(repo_path("data", "sample", "customer_segments.csv"))
feats9 = ["annual_income_musd", "spending_score", "age", "visits_per_month"]
### START CODE HERE ### (≈ 10 dòng)
# STARTER: X9 = None             # 4 feature đã chuẩn hoá
# STARTER: k9 = None             # K ∈ {3..8} có silhouette lớn nhất
# STARTER: profile9 = None       # trung bình feature theo cụm + cột n (số khách)
# STARTER: segment_names = None  # list tên cụm (theo thứ tự cluster 0, 1, ...)
# STARTER: ari9 = None
X9 = StandardScaler().fit_transform(seg[feats9])
sil9 = {k: silhouette_score(X9, KMeans(k, n_init=10, random_state=0).fit_predict(X9)) for k in range(3, 9)}
k9 = max(sil9, key=sil9.get)
seg["cluster"] = KMeans(k9, n_init=10, random_state=0).fit_predict(X9)
profile9 = seg.groupby("cluster")[feats9].mean().round(1).assign(n=seg.cluster.value_counts().sort_index())


def name_cluster(row):
    rich, spend, young = row.annual_income_musd > 60, row.spending_score > 50, row.age < 30
    return {(True, True): "chi tiêu mạnh", (True, False): "giàu thận trọng", (False, True): "trẻ bốc đồng" if young else "trung lưu",
            (False, False): "tiết kiệm"}[(rich, spend)]


segment_names = [name_cluster(r) for _, r in profile9.iterrows()]
ari9 = adjusted_rand_score(seg.true_segment, seg.cluster)
### END CODE HERE ###
print(f"K = {k9} | ARI = {ari9:.3f}")
profile9.assign(tên=segment_names)

In [ ]:
check_greater(ari9, 0.85, "ARI ≥ 0.85")
check_equal(len(segment_names), k9, "mỗi cụm một tên")

## L7 — Nâng cao

**Bài 10.** **Độ ổn định của clustering:** với mỗi K ∈ {2..7}, chạy K-Means trên 20 cặp mẫu bootstrap của dữ liệu khách hàng (đã scale), gán nhãn cho **toàn bộ** dữ liệu bằng hai model, tính ARI giữa hai cách gán; độ ổn định = ARI trung bình. Lưu Series `stability` (index = K). K "thật" thường có độ ổn định cao. Ứng viên tốt cho K là gì?

In [ ]:
### START CODE HERE ### (≈ 9 dòng)
# STARTER: # với mỗi K: 20 lần — fit hai KMeans trên hai mẫu bootstrap khác nhau của X9, predict trên toàn bộ X9, ARI giữa hai cách gán
# STARTER: stability = None     # Series: K → ARI trung bình
stab = {}
for k in range(2, 8):
    aris = []
    for b in range(20):
        r = np.random.default_rng(b)
        i1, i2 = r.integers(0, len(X9), len(X9)), r.integers(0, len(X9), len(X9))
        m1 = KMeans(k, n_init=5, random_state=b).fit(X9[i1])
        m2 = KMeans(k, n_init=5, random_state=b + 100).fit(X9[i2])
        aris.append(adjusted_rand_score(m1.predict(X9), m2.predict(X9)))
    stab[k] = np.mean(aris)
stability = pd.Series(stab)
### END CODE HERE ###
print(stability.round(3))

In [ ]:
check_greater(stability.loc[5], 0.85, "K = 5 (số phân khúc thật) rất ổn định")
check_greater(stability.loc[5], stability.loc[7], "K = 5 ổn định hơn K = 7")

✍️ K nào là ứng viên tốt?

**Gợi ý:** K = 2 ổn định nhất — chia đôi thô gần như luôn giống nhau, nên độ ổn định **thiên vị K nhỏ** và không thể dùng một mình. Điều đáng chú ý là K = 5 ổn định hơn hẳn các K lân cận (4 và 6): một "đỉnh" ổn định cục bộ như vậy là dấu hiệu cấu trúc thật. Kết hợp với silhouette (cũng cao nhất ở K = 5) và khả năng diễn giải profile → chọn K = 5.